# 模块2｜Function Calling - 健康数据管理工具

**核心技术**：Gemini的Function Calling功能

**目标**：让AI调用工具函数来执行外部操作（保存数据、查询记录、设置提醒等）

---

##  本模块学习目标

在这个模块中，我们将：
1. ✅ 理解Function Calling的作用和适用场景
2. ✅ 声明工具函数的schema
3. ✅ 让AI自动选择并调用合适的工具
4. ✅ 处理函数调用结果
5. ✅ 实现多轮Function Calling

---

## 为什么需要Function Calling？

**设计原则**：工具用于外部操作，判断和分析由LLM完成

### ✅ 适合用Function的场景：
- 保存数据到"数据库"
- 查询历史记录
- 设置提醒/通知
- 记录日志

### ❌ 不适合用Function的场景：
- BMI计算（LLM可以直接算）
- 健康评分（LLM直接分析）
- 生成建议（LLM的核心能力）

## 1️⃣ 环境配置

首先，安装必要的库并配置API密钥

⚠️ **数据安全提醒**：使用Gemini API时数据会发送到Google服务器，建议使用示例数据

In [1]:
# 安装依赖（使用新的google-genai SDK）
%pip install -q google-genai

In [2]:
import os
from google import genai
from google.genai import types

# 如果在 Colab 中运行
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except ImportError:
    # 如果在本地运行，从环境变量读取
    api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    raise ValueError("❌ 请先配置 GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("✅ Gemini API 配置成功！")

✅ Gemini API 配置成功！


## 2️⃣ 理解模块1的输出数据

在模块1中，我们已经将小明的自然语言健康描述转换为结构化JSON数据。在模块2中，我们将使用这些结构化数据，模拟它们已经被"存储"在系统中。

In [3]:
# 这是从模块1输出的结构化数据（小明的两天健康记录）
# 在真实应用中，这些数据会存储在数据库中
stored_health_data = {
    "2025-11-18": {
        "date": "2025-11-18",
        "sleep": {
            "bedtime": "23:00",
            "wake_time": "07:00",
            "duration_hours": 8,
            "quality": "良好",
            "notes": "中间醒了一次"
        },
        "meals": [
            {
                "type": "早餐",
                "time": "08:00",
                "foods": ["煎饼果子", "豆浆"],
                "calories": 450
            },
            {
                "type": "午餐",
                "time": "12:30",
                "foods": ["盖浇饭", "青菜"],
                "calories": 680
            },
            {
                "type": "晚餐",
                "time": "19:00",
                "foods": ["鸡胸肉", "西兰花", "糙米饭"],
                "calories": 520
            }
        ],
        "exercise": [
            {
                "type": "跑步",
                "duration_minutes": 30,
                "intensity": "中等",
                "time": "18:00"
            }
        ],
        "mood": "良好",
        "stress_level": "中等",
        "notes": "今天工作比较忙"
    },
    "2025-11-19": {
        "date": "2025-11-19",
        "sleep": {
            "bedtime": "00:30",
            "wake_time": "07:30",
            "duration_hours": 7,
            "quality": "一般",
            "notes": "睡前刷手机了"
        },
        "meals": [
            {
                "type": "早餐",
                "time": "08:30",
                "foods": ["面包", "牛奶"],
                "calories": 350
            },
            {
                "type": "午餐",
                "time": "13:00",
                "foods": ["外卖炒饭"],
                "calories": 750
            },
            {
                "type": "晚餐",
                "time": "20:00",
                "foods": ["火锅"],
                "calories": 900
            }
        ],
        "exercise": [],
        "mood": "一般",
        "stress_level": "较高",
        "notes": "今天加班到很晚"
    }
}

print("📊 已加载小明的健康数据（共{}天）".format(len(stored_health_data)))

📊 已加载小明的健康数据（共2天）


## 3️⃣ 实现模拟工具函数

这些函数模拟真实应用中的外部操作（数据库、API调用等）

In [4]:
from datetime import datetime

# 模拟数据库
health_records_db = stored_health_data.copy()  # 从模块1继承的数据
reminders_db = []
symptoms_db = []
user_goals = {
    "sleep_hours": 8,
    "exercise_minutes_per_week": 150,
    "water_intake_ml": 2000,
    "weight_goal_kg": 70
}

# 工具函数1：保存健康记录
def save_health_record(date: str, sleep_hours: float, meals: str, exercise: str, mood: str) -> dict:
    """保存健康记录到数据库（模拟）

    Args:
        date: 日期，格式：YYYY-MM-DD
        sleep_hours: 睡眠时长（小时）
        meals: 一天的饮食描述
        exercise: 运动情况描述
        mood: 心情状态

    Returns:
        包含状态、记录ID和消息的字典
    """
    print(f"🔧 [工具调用] save_health_record - 保存健康记录：日期={date}, 睡眠={sleep_hours}h, 饮食={meals[:20]}..., 运动={exercise[:20]}..., 心情={mood}")

    record_id = f"{date}-{len(health_records_db) + 1:03d}"

    health_records_db[date] = {
        "record_id": record_id,
        "date": date,
        "sleep_hours": sleep_hours,
        "meals": meals,
        "exercise": exercise,
        "mood": mood,
        "created_at": datetime.now().isoformat()
    }

    return {
        "status": "success",
        "record_id": record_id,
        "message": f"健康记录已保存（日期：{date}）"
    }

# 工具函数2：查询历史记录
def get_historical_records(start_date: str, end_date: str) -> dict:
    """查询指定日期范围的健康记录

    Args:
        start_date: 开始日期，格式：YYYY-MM-DD
        end_date: 结束日期，格式：YYYY-MM-DD

    Returns:
        包含状态、数量和记录列表的字典
    """
    print(f"🔧 [工具调用] get_historical_records - 查询历史记录：{start_date} 至 {end_date}")

    matching_records = []

    for date, record in health_records_db.items():
        if start_date <= date <= end_date:
            matching_records.append(record)

    return {
        "status": "success",
        "count": len(matching_records),
        "records": matching_records
    }

# 工具函数3：查询食物营养信息
def query_food_database(food_name: str) -> dict:
    """查询食物的营养成分信息（卡路里、蛋白质、脂肪、碳水化合物）

    Args:
        food_name: 食物名称

    Returns:
        包含食物营养信息的字典
    """
    print(f"🔧 [工具调用] query_food_database - 查询食物营养信息：{food_name}")

    # 模拟食物数据库
    food_db = {
        "鸡蛋": {"calories": 70, "protein": 6, "fat": 5, "carbs": 0.6},
        "牛奶": {"calories": 150, "protein": 8, "fat": 8, "carbs": 12},
        "煎饼果子": {"calories": 350, "protein": 12, "fat": 15, "carbs": 45},
        "盖浇饭": {"calories": 680, "protein": 25, "fat": 20, "carbs": 90},
        "鸡胸肉": {"calories": 165, "protein": 31, "fat": 3.6, "carbs": 0},
        "西兰花": {"calories": 55, "protein": 4.5, "fat": 0.6, "carbs": 11},
        "糙米饭": {"calories": 110, "protein": 2.6, "fat": 0.9, "carbs": 23}
    }

    if food_name in food_db:
        return {
            "status": "success",
            "food_name": food_name,
            "nutrition": food_db[food_name]
        }
    else:
        return {
            "status": "not_found",
            "message": f"未找到'{food_name}'的营养信息"
        }

# 工具函数4：设置健康提醒
def set_health_reminder(time: str, message: str, reminder_type: str) -> dict:
    """为用户设置健康相关的提醒（如喝水、运动、睡觉）

    Args:
        time: 提醒时间，格式：HH:MM
        message: 提醒内容
        reminder_type: 提醒类型（如：drink_water, exercise, sleep）

    Returns:
        包含提醒ID和确认消息的字典
    """
    print(f"🔧 [工具调用] set_health_reminder - 设置提醒：时间={time}, 类型={reminder_type}, 内容={message}")

    reminder_id = f"reminder-{len(reminders_db) + 1:04d}"

    reminder = {
        "reminder_id": reminder_id,
        "time": time,
        "message": message,
        "type": reminder_type,
        "created_at": datetime.now().isoformat(),
        "status": "active"
    }

    reminders_db.append(reminder)

    return {
        "status": "success",
        "reminder_id": reminder_id,
        "message": f"提醒已设置：{time} - {message}"
    }

# 工具函数5：记录症状
def log_symptom(date: str, symptom: str, severity: str, notes: str = "") -> dict:
    """记录用户的健康症状或不适

    Args:
        date: 症状出现日期，格式：YYYY-MM-DD
        symptom: 症状描述
        severity: 严重程度（轻微/中等/严重）
        notes: 额外备注（可选）

    Returns:
        包含症状ID和确认消息的字典
    """
    print(f"🔧 [工具调用] log_symptom - 记录症状：日期={date}, 症状={symptom}, 严重程度={severity}")

    symptom_id = f"symptom-{len(symptoms_db) + 1:04d}"

    symptom_log = {
        "symptom_id": symptom_id,
        "date": date,
        "symptom": symptom,
        "severity": severity,
        "notes": notes,
        "logged_at": datetime.now().isoformat()
    }

    symptoms_db.append(symptom_log)

    return {
        "status": "success",
        "symptom_id": symptom_id,
        "message": f"症状已记录：{symptom}（{severity}）"
    }

# 工具函数6：获取用户健康目标
def get_user_health_goals() -> dict:
    """获取用户设定的健康目标（如每天睡眠时长、每周运动时间等）

    Returns:
        包含用户健康目标的字典
    """
    print(f"🔧 [工具调用] get_user_health_goals - 获取用户健康目标")

    return {
        "status": "success",
        "goals": user_goals
    }

print("✅ 所有工具函数已定义！")

✅ 所有工具函数已定义！


## 4️⃣ 配置模型工具

使用新的SDK，我们只需要将函数直接传给配置即可！SDK会自动从函数的类型提示和docstring中提取信息。

In [5]:
# 创建配置，直接传入函数对象
# SDK会自动从类型提示和docstring中提取schema信息
config = types.GenerateContentConfig(
    tools=[
        save_health_record,
        get_historical_records,
        query_food_database,
        set_health_reminder,
        log_symptom,
        get_user_health_goals
    ]
)

print("✅ 工具配置已完成！")
print("💡 注意：新SDK会自动从函数的类型提示和docstring中提取信息，无需手动声明schema")

✅ 工具配置已完成！
💡 注意：新SDK会自动从函数的类型提示和docstring中提取信息，无需手动声明schema


## 5️⃣ 示例1：保存健康记录

用户用自然语言描述今天的健康状况，AI自动调用工具保存

In [6]:
# 用户输入（自然语言）
user_message = """
今天是2025年11月22日，我昨晚11点半睡的，今早7点醒，睡了7.5小时。
早餐吃了鸡蛋和牛奶，午饭吃了盖浇饭，晚饭吃了鸡胸肉和西兰花。
晚上跑步了30分钟，心情不错。
帮我记录一下。
"""

print("👤 用户：", user_message)
print("\n" + "="*50)

# 使用新SDK调用 - SDK自动处理函数调用！
response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents=user_message,
    config=config
)

print(f"🤖 AI回复: {response.text}")
print("\n💡 注意：SDK自动处理了函数调用，我们直接得到了最终结果！")
print("-"*80)
print(f"已保存数据 health_records_db: {health_records_db}")

👤 用户： 
今天是2025年11月22日，我昨晚11点半睡的，今早7点醒，睡了7.5小时。
早餐吃了鸡蛋和牛奶，午饭吃了盖浇饭，晚饭吃了鸡胸肉和西兰花。
晚上跑步了30分钟，心情不错。
帮我记录一下。


🔧 [工具调用] save_health_record - 保存健康记录：日期=2025-11-22, 睡眠=7.5h, 饮食=早餐：鸡蛋和牛奶，午餐：盖浇饭，晚餐：鸡..., 运动=跑步30分钟..., 心情=不错
🤖 AI回复: 好的，已经帮您记录了2025年11月22日的健康数据：
- **睡眠**：7.5小时
- **饮食**：早餐鸡蛋和牛奶，午餐盖浇饭，晚餐鸡胸肉和西兰花
- **运动**：跑步30分钟
- **心情**：不错

记录ID为 2025-11-22-003。保持良好的生活习惯，继续加油！

💡 注意：SDK自动处理了函数调用，我们直接得到了最终结果！
--------------------------------------------------------------------------------
已保存数据 health_records_db: {'2025-11-18': {'date': '2025-11-18', 'sleep': {'bedtime': '23:00', 'wake_time': '07:00', 'duration_hours': 8, 'quality': '良好', 'notes': '中间醒了一次'}, 'meals': [{'type': '早餐', 'time': '08:00', 'foods': ['煎饼果子', '豆浆'], 'calories': 450}, {'type': '午餐', 'time': '12:30', 'foods': ['盖浇饭', '青菜'], 'calories': 680}, {'type': '晚餐', 'time': '19:00', 'foods': ['鸡胸肉', '西兰花', '糙米饭'], 'calories': 520}], 'exercise': [{'type': '跑步', 'duration_minutes': 30, 'intensity': '中等', 'time': '18:00'}], 'mood': '良好', 'stre

## 6️⃣ 示例2：查询历史记录并分析

AI会自动调用查询工具，然后**自己分析数据**（不调用分析函数）

In [7]:
# 用户询问
user_message = "我这两天（2025年11月18日到20日）的睡眠怎么样？给我分析一下"

print("👤 用户：", user_message)
print("\n" + "="*50)

# SDK自动处理查询和分析
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_message,
    config=config
)

print(f"🤖 AI分析: {response.text}")
print("\n💡 注意：数据查询由工具完成，数据分析由LLM直接完成！")

👤 用户： 我这两天（2025年11月18日到20日）的睡眠怎么样？给我分析一下

🔧 [工具调用] get_historical_records - 查询历史记录：2025-11-18 至 2025-11-20
🤖 AI分析: 根据您提供的信息，我为您分析了2025年11月18日和19日的睡眠情况，20日没有相关记录：

*   **11月18日**：您睡了8小时，睡眠质量良好，但中途醒来一次。
*   **11月19日**：您睡了7小时，睡眠质量一般，睡前有玩手机的习惯。

**总结**：

在18日您有充足的睡眠时间且质量良好，但在19日您的睡眠时间略有减少，并且睡眠质量下降，这可能与您睡前使用手机有关。建议您在睡前避免使用电子产品，以帮助提高睡眠质量。

💡 注意：数据查询由工具完成，数据分析由LLM直接完成！


## 7️⃣ 示例3：多轮Function Calling

AI可能需要调用多个工具来完成一个任务 - SDK会自动处理所有调用！

In [8]:
# 测试多轮function calling
user_message = """
我想查一下鸡蛋的营养成分，然后帮我设置一个明天早上8点的早餐提醒。
"""

print("👤 用户：", user_message)
print("="*50)

response = client.models.generate_content(
    model="gemini-2.5-pro",
    contents=user_message,
    config=config
)

print(f"\n✅ 最终回复:\n{response.text}")
print("\n💡 观察：AI自动决定了调用顺序和次数，SDK处理了所有细节！")

👤 用户： 
我想查一下鸡蛋的营养成分，然后帮我设置一个明天早上8点的早餐提醒。

🔧 [工具调用] query_food_database - 查询食物营养信息：鸡蛋
🔧 [工具调用] set_health_reminder - 设置提醒：时间=08:00, 类型=breakfast, 内容=该吃早餐了

✅ 最终回复:
好的，已经为您查询到鸡蛋的营养成分，并设置了明天早上8点的早餐提醒。

一个普通大小的鸡蛋大约含有：
*   卡路里：70
*   蛋白质：6克
*   脂肪：5克
*   碳水化合物：0.6克


💡 观察：AI自动决定了调用顺序和次数，SDK处理了所有细节！


## 8️⃣ 示例4：AI判断何时使用工具

让我们看一个例子：AI直接计算 vs 调用工具

In [11]:
# 场景1：AI可以直接计算的任务
user_message_1 = "我身高175cm，体重70kg，帮我算一下BMI"

print("👤 用户：", user_message_1)
print("="*50)

response = client.models.generate_content(
    model="gemini-2.5-pro",
    contents=user_message_1,
    config=config
)

print(f"🤖 AI回复（直接计算）: {response.text}")

print("\n" + "="*50 + "\n")

# 场景2：需要查询外部数据的任务
user_message_2 = "西兰花的营养成分是什么？"

print("👤 用户：", user_message_2)
print("="*50)

response = client.models.generate_content(
    model="gemini-2.5-pro",
    contents=user_message_2,
    config=config
)

print(f"\n🤖 AI回复（调用工具查询）: {response.text}")

print("\n💡 关键区别：")
print("   - BMI计算：AI直接算（不需要外部数据）")
print("   - 营养成分：AI调用工具（需要查询数据库）")

👤 用户： 我身高175cm，体重70kg，帮我算一下BMI
🤖 AI回复（直接计算）: 我目前没有内置的BMI计算工具，但是我可以根据你提供的信息为你计算：

BMI的计算公式是：**体重（kg） / [身高（m）]²**

1.  **将身高从厘米转换为米**: 175cm = 1.75m
2.  **计算身高的平方**: 1.75 \* 1.75 = 3.0625
3.  **用体重除以身高的平方**: 70 / 3.0625 ≈ 22.86

所以，你的BMI大约是 **22.86**。

根据通常的标准，这个值处于**正常体重**范围内（18.5 - 24.9）。


👤 用户： 西兰花的营养成分是什么？
🔧 [工具调用] query_food_database - 查询食物营养信息：西兰花

🤖 AI回复（调用工具查询）: 好的，西兰花的营养成分如下：

*   **热量:** 55大卡
*   **蛋白质:** 4.5克
*   **脂肪:** 0.6克
*   **碳水化合物:** 11克

💡 关键区别：
   - BMI计算：AI直接算（不需要外部数据）
   - 营养成分：AI调用工具（需要查询数据库）


## 9️⃣ 查看我们的"数据库"

让我们看看通过Function Calling保存的数据

In [10]:
print("📊 健康记录数据库：")
print(f"   总记录数：{len(health_records_db)}")
for date in sorted(health_records_db.keys()):
    print(f"   - {date}")

print(f"\n⏰ 提醒列表：")
if reminders_db:
    for reminder in reminders_db:
        print(f"   - [{reminder['time']}] {reminder['message']}")
else:
    print("   （暂无提醒）")

print(f"\n🏥 症状记录：")
if symptoms_db:
    for symptom in symptoms_db:
        print(f"   - [{symptom['date']}] {symptom['symptom']} ({symptom['severity']})")
else:
    print("   （暂无症状记录）")

print(f"\n🎯 用户健康目标：")
for key, value in user_goals.items():
    print(f"   - {key}: {value}")

print("\n💡 通过查看数据库，我们可以确认函数调用成功保存了数据！")

📊 健康记录数据库：
   总记录数：3
   - 2025-11-18
   - 2025-11-19
   - 2025-11-22

⏰ 提醒列表：
   - [08:00] 该吃早餐了

🏥 症状记录：
   （暂无症状记录）

🎯 用户健康目标：
   - sleep_hours: 8
   - exercise_minutes_per_week: 150
   - water_intake_ml: 2000
   - weight_goal_kg: 70

💡 通过查看数据库，我们可以确认函数调用成功保存了数据！
